In [1]:
# ══════════════════════════════════════════════════════════
# CELDA 1 — Librerías y rutas
# ══════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import geopandas as gpd
import json
import warnings
warnings.filterwarnings('ignore')

BASE      = '/content/drive/MyDrive/TFM_Seguridad_Vial'
outputs   = f'{BASE}/outputs'
shp_path  = f'{BASE}/datos/distritos/raw/BARRIOS.shp'
pbi_path  = f'{BASE}/outputs/PowerBI'

import os
os.makedirs(pbi_path, exist_ok=True)

print('✅ Rutas configuradas')
print(f'Output PowerBI: {pbi_path}')

Mounted at /content/drive
✅ Rutas configuradas
Output PowerBI: /content/drive/MyDrive/TFM_Seguridad_Vial/outputs/PowerBI


In [2]:
# ══════════════════════════════════════════════════════════
# CELDA 2 — Generar GeoJSON de distritos
# ══════════════════════════════════════════════════════════

# Cargamos shapefile de barrios y disolvemos por distrito
gdf = gpd.read_file(shp_path)
print(f'Shapefile cargado: {gdf.shape}')
print(f'Columnas: {gdf.columns.tolist()}')
print(f'CRS original: {gdf.crs}')
print(f'\nPrimeras filas:')
print(gdf.head(3).to_string())

Shapefile cargado: (131, 12)
Columnas: ['CODDIS', 'NOMDIS', 'COD_BAR', 'NOMBRE', 'COD_DIS_TX', 'BARRIO_MAY', 'COD_DISBAR', 'NUM_BAR', 'BARRIO_MT', 'COD_DISB', 'Area', 'geometry']
CRS original: EPSG:25830

Primeras filas:
  CODDIS  NOMDIS COD_BAR       NOMBRE COD_DIS_TX   BARRIO_MAY COD_DISBAR NUM_BAR    BARRIO_MT COD_DISB          Area                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [3]:
# ══════════════════════════════════════════════════════════
# CELDA 3 — Disolver barrios → distritos y exportar GeoJSON
# ══════════════════════════════════════════════════════════

# Identificamos la columna de nombre de distrito
# (puede ser NOMDIS, NOMBRE, DISTRITO, etc.)
col_distrito = None
for col in gdf.columns:
    if any(kw in col.upper() for kw in ['NOMDIS', 'DISTRITO', 'DENOM']):
        col_distrito = col
        print(f'Columna de distrito encontrada: {col_distrito}')
        print(f'Valores únicos: {sorted(gdf[col_distrito].unique())}')
        break

if col_distrito is None:
    print('Columnas disponibles:', gdf.columns.tolist())
    print('⚠️ Especifica manualmente col_distrito')

Columna de distrito encontrada: NOMDIS
Valores únicos: ['Arganzuela', 'Barajas', 'Carabanchel', 'Centro', 'Chamartín', 'Chamberí', 'Ciudad Lineal', 'Fuencarral - El Pardo', 'Hortaleza', 'Latina', 'Moncloa - Aravaca', 'Moratalaz', 'Puente de Vallecas', 'Retiro', 'Salamanca', 'San Blas - Canillejas', 'Tetuán', 'Usera', 'Vicálvaro', 'Villa de Vallecas', 'Villaverde']


In [4]:
# ══════════════════════════════════════════════════════════
# CELDA 4 — Crear GeoJSON final para Power BI
# (ejecutar después de confirmar col_distrito en celda 3)
# ══════════════════════════════════════════════════════════

# Ajusta col_distrito si es necesario tras ver celda 3
# col_distrito = 'NOMDIS'  # ← descomenta y ajusta si hace falta

# Normalizamos nombres de distrito
gdf['DISTRITO_NORM'] = (gdf[col_distrito]
    .str.strip().str.upper()
    .str.replace(' - ', '-', regex=False)
    .str.replace('CHAMARTIN', 'CHAMARTÍN', regex=False)
    .str.replace('CHAMBERI', 'CHAMBERÍ', regex=False)
    .str.replace('TETUAN', 'TETUÁN', regex=False)
    .str.replace('VICALVARO', 'VICÁLVARO', regex=False)
    .str.replace('SAN BLAS$', 'SAN BLAS-CANILLEJAS', regex=True)
)

# Disolvemos barrios → distritos
gdf_distritos = gdf.dissolve(by='DISTRITO_NORM').reset_index()
gdf_distritos = gdf_distritos[['DISTRITO_NORM', 'geometry']].copy()
gdf_distritos.columns = ['distrito', 'geometry']

# Proyectamos a WGS84 (necesario para Power BI)
gdf_distritos = gdf_distritos.to_crs(epsg=4326)

print(f'Distritos generados: {len(gdf_distritos)}')
print(f'CRS final: {gdf_distritos.crs}')
print(f'Distritos: {sorted(gdf_distritos["distrito"].tolist())}')

# Verificación: deben ser exactamente 21
assert len(gdf_distritos) == 21, f'⚠️ Se esperaban 21 distritos, hay {len(gdf_distritos)}'

# Exportamos GeoJSON
geojson_path = f'{pbi_path}/distritos_madrid.geojson'
gdf_distritos.to_file(geojson_path, driver='GeoJSON')
print(f'\n✅ GeoJSON guardado: {geojson_path}')

Distritos generados: 21
CRS final: EPSG:4326
Distritos: ['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CENTRO', 'CHAMARTÍN', 'CHAMBERÍ', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'HORTALEZA', 'LATINA', 'MONCLOA-ARAVACA', 'MORATALAZ', 'PUENTE DE VALLECAS', 'RETIRO', 'SALAMANCA', 'SAN BLAS-CANILLEJAS', 'TETUÁN', 'USERA', 'VICÁLVARO', 'VILLA DE VALLECAS', 'VILLAVERDE']

✅ GeoJSON guardado: /content/drive/MyDrive/TFM_Seguridad_Vial/outputs/PowerBI/distritos_madrid.geojson


In [5]:
# ══════════════════════════════════════════════════════════
# CELDA 5 — CSV principal: Score Final + IPD + IRP
# ══════════════════════════════════════════════════════════

# Cargamos todos los índices
score_2025 = pd.read_csv(f'{outputs}/score_final.csv')
score_2025.columns = ['posicion_2025', 'distrito', 'IPD_medio', 'IPD_norm',
                       'IRP_2025', 'Score_2025']

score_2026 = pd.read_csv(f'{outputs}/score_final_2026.csv')
score_2026.columns = ['posicion_2026', 'distrito', 'IPD_medio_26', 'IPD_norm_26',
                       'acc_pred_2026', 'IRP_2026', 'Score_2026']

ipd_completo = pd.read_csv(f'{outputs}/IPD_completo.csv')

# Tabla maestra anual para variables demográficas
tabla_anual = pd.read_csv(f'{outputs}/tabla_anual_final.csv')
demo_2024 = tabla_anual[tabla_anual['año'] == 2024][[
    'distrito', 'poblacion', 'pct_mayores_65', 'area_km2',
    'km_carril_bici', 'n_paradas_emt'
]].copy()

# Unimos todo
df_pbi = score_2025[['distrito', 'IPD_norm', 'IRP_2025', 'Score_2025', 'posicion_2025']].merge(
    score_2026[['distrito', 'IRP_2026', 'Score_2026', 'posicion_2026']],
    on='distrito', how='left'
).merge(
    demo_2024, on='distrito', how='left'
)

# IPD medio anual (para evolución temporal)
ipd_medio = ipd_completo.groupby('distrito')['IPD'].mean().reset_index()
ipd_medio.columns = ['distrito', 'IPD_medio']
df_pbi = df_pbi.merge(ipd_medio, on='distrito', how='left')

# Variables calculadas
df_pbi['cambio_ranking'] = df_pbi['posicion_2025'] - df_pbi['posicion_2026']
df_pbi['perfil'] = pd.cut(
    df_pbi['Score_2025'],
    bins=[-1, 30, 50, 70, 101],
    labels=['Seguimiento periódico', 'Intervención de consolidación',
            'Intervención preventiva', 'Intervención urgente']
)

# Ordenamos por Score_2025
df_pbi = df_pbi.sort_values('Score_2025', ascending=False).reset_index(drop=True)
df_pbi['posicion_final'] = range(1, len(df_pbi) + 1)

print(f'Forma: {df_pbi.shape}')
print(f'Columnas: {df_pbi.columns.tolist()}')
print(df_pbi[['distrito', 'Score_2025', 'Score_2026', 'perfil']].to_string())

df_pbi.to_csv(f'{pbi_path}/pbi_score_ranking.csv',
              index=False, encoding='utf-8-sig')
print('\n✅ pbi_score_ranking.csv guardado')

Forma: (21, 17)
Columnas: ['distrito', 'IPD_norm', 'IRP_2025', 'Score_2025', 'posicion_2025', 'IRP_2026', 'Score_2026', 'posicion_2026', 'poblacion', 'pct_mayores_65', 'area_km2', 'km_carril_bici', 'n_paradas_emt', 'IPD_medio', 'cambio_ranking', 'perfil', 'posicion_final']
               distrito  Score_2025  Score_2026                         perfil
0                CENTRO       89.32       89.32           Intervención urgente
1              CHAMBERÍ       82.47       83.40           Intervención urgente
2             SALAMANCA       76.31       78.21           Intervención urgente
3                RETIRO       71.42       72.24           Intervención urgente
4    PUENTE DE VALLECAS       63.17       62.61        Intervención preventiva
5             CHAMARTÍN       60.61       60.96        Intervención preventiva
6                TETUÁN       60.38       61.09        Intervención preventiva
7         CIUDAD LINEAL       60.26       61.57        Intervención preventiva
8              

In [6]:
# ══════════════════════════════════════════════════════════
# CELDA 6 — CSV evolución temporal IPD (2021-2024)
# ══════════════════════════════════════════════════════════

ipd_completo = pd.read_csv(f'{outputs}/IPD_completo.csv')

# Seleccionamos columnas clave
cols = ['distrito', 'año', 'IPD', 'dim_seguridad', 'dim_vulnerabilidad',
        'acc_ponderado', 'acc_pond_km2', 'tasa_mortalidad',
        'pct_mayores_65', 'densidad_vulnerable']
cols_disponibles = [c for c in cols if c in ipd_completo.columns]
ipd_pbi = ipd_completo[cols_disponibles].copy()

# Añadimos ranking por año
ipd_pbi['rank_ipd_año'] = ipd_pbi.groupby('año')['IPD'] \
    .rank(ascending=False, method='min').astype(int)

ipd_pbi.to_csv(f'{pbi_path}/pbi_ipd_temporal.csv',
               index=False, encoding='utf-8-sig')
print(f'✅ pbi_ipd_temporal.csv guardado — {ipd_pbi.shape}')
print(ipd_pbi.head(5).to_string())

✅ pbi_ipd_temporal.csv guardado — (84, 10)
     distrito   año    IPD  dim_seguridad  dim_vulnerabilidad  acc_pond_km2  tasa_mortalidad  pct_mayores_65  densidad_vulnerable  rank_ipd_año
0  ARGANZUELA  2021  40.45        25.7588             58.4134       28.7926              0.0           19.83              7343.64             7
1  ARGANZUELA  2022  41.78        25.5778             61.5811       21.9814              0.0           20.45              7477.72            10
2  ARGANZUELA  2023  42.97        25.6903             64.0888       23.8390              0.0           20.87              7658.30            10
3  ARGANZUELA  2024  47.65        33.2571             65.2415       26.4706              0.0           21.07              7734.02             9
4     BARAJAS  2021  16.23        11.3514             22.1890        1.4382              0.0           17.89               397.84            19


In [7]:
# ══════════════════════════════════════════════════════════
# CELDA 7 — CSV validación IRP (predicho vs real 2025)
# ══════════════════════════════════════════════════════════

val = pd.read_csv(f'{outputs}/validacion_IRP_2025.csv')
print(f'Columnas validación: {val.columns.tolist()}')
print(val.head(3).to_string())

# Limpiamos y guardamos
val.to_csv(f'{pbi_path}/pbi_validacion_irp.csv',
           index=False, encoding='utf-8-sig')
print(f'\n✅ pbi_validacion_irp.csv guardado — {val.shape}')

Columnas validación: ['posicion', 'distrito', 'Score_final', 'acc_pred_2025', 'IRP', 'acc_vru_real_2025', 'acc_pond_real_2025', 'error_abs', 'error_pct', 'rank_real_2025', 'rank_pred_2025', 'diff_rank']
   posicion   distrito  Score_final  acc_pred_2025     IRP  acc_vru_real_2025  acc_pond_real_2025  error_abs  error_pct  rank_real_2025  rank_pred_2025  diff_rank
0         1     CENTRO        89.32         294.32  100.00                248                 245       49.3       20.1               2               1          1
1         2   CHAMBERÍ        82.47         189.74   56.17                140                 202      -12.3       -6.1               6               7          1
2         3  SALAMANCA        76.31         197.55   59.45                171                 248      -50.4      -20.3               1               5          4

✅ pbi_validacion_irp.csv guardado — (21, 12)


In [8]:
# ══════════════════════════════════════════════════════════
# CELDA 8 — CSV análisis horario Top 3
# ══════════════════════════════════════════════════════════

# Reconstruimos el análisis horario desde los datos raw
import os

BASE    = '/content/drive/MyDrive/TFM_Seguridad_Vial'
raw     = f'{BASE}/datos/accidentes_general/raw'
TOP3    = ['CENTRO', 'CHAMBERÍ', 'SALAMANCA']

mapa_gravedad = {
    1: 'leve', 2: 'leve', 5: 'leve', 6: 'leve', 7: 'leve',
    3: 'grave', 4: 'fallecido', 14: 'sin_asistencia', 77: 'desconocido'
}

def quitar_acentos(s):
    return (s.str.replace('á','a',regex=False).str.replace('é','e',regex=False)
             .str.replace('í','i',regex=False).str.replace('ó','o',regex=False)
             .str.replace('ú','u',regex=False))

dfs = []
for año in range(2019, 2026):
    try:
        df = pd.read_csv(f'{raw}/Accidentes{año}.csv', sep=';', encoding='utf-8-sig')
        df.columns = (df.columns.str.strip().str.lower()
                      .str.replace('ó','o',regex=False).str.replace('é','e',regex=False)
                      .str.replace('í','i',regex=False).str.replace(' ','_',regex=False))
        df['año'] = año
        dfs.append(df)
    except Exception as e:
        print(f'⚠️ {año}: {e}')

df_raw = pd.concat(dfs, ignore_index=True)
df_raw['distrito'] = df_raw['distrito'].str.strip().str.upper().str.replace(' - ','-',regex=False)
df_raw['hora_int'] = pd.to_datetime(df_raw['hora'], format='%H:%M:%S', errors='coerce').dt.hour
df_raw['fecha_dt'] = pd.to_datetime(df_raw['fecha'], dayfirst=True, errors='coerce')
df_raw['dia_semana'] = df_raw['fecha_dt'].dt.day_name()
df_raw['mes'] = df_raw['fecha_dt'].dt.month
df_raw['gravedad'] = df_raw['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')

tipo_persona_norm  = quitar_acentos(df_raw['tipo_persona'].str.strip().str.lower())
tipo_vehiculo_norm = quitar_acentos(df_raw['tipo_vehiculo'].str.strip().str.lower())
vru_mask = (tipo_persona_norm == 'peaton') | (tipo_vehiculo_norm.str.contains('bici', na=False))
df_vru = df_raw[vru_mask].copy()
df_vru['tipo_vru'] = np.where(tipo_vehiculo_norm[vru_mask].str.contains('bici', na=False), 'Ciclista', 'Peatón')

df_top3 = df_vru[df_vru['distrito'].isin(TOP3)].copy()

# Agregamos por hora
horario = (
    df_top3.groupby(['distrito', 'hora_int', 'tipo_vru'])
    ['num_expediente'].nunique()
    .reset_index()
    .rename(columns={'num_expediente': 'n_accidentes', 'hora_int': 'hora'})
)
horario.to_csv(f'{pbi_path}/pbi_horario_top3.csv', index=False, encoding='utf-8-sig')

# Agregamos por día de semana
diasem = (
    df_top3.groupby(['distrito', 'dia_semana', 'tipo_vru'])
    ['num_expediente'].nunique()
    .reset_index()
    .rename(columns={'num_expediente': 'n_accidentes'})
)
diasem.to_csv(f'{pbi_path}/pbi_diasemana_top3.csv', index=False, encoding='utf-8-sig')

# Calles más peligrosas
df_top3['calle'] = (df_top3['localizacion'].str.strip().str.upper()
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.replace(r'\d+$', '', regex=True).str.strip())
calles = (
    df_top3.groupby(['distrito', 'calle'])
    .agg(n_accidentes=('num_expediente', 'nunique'),
         n_graves=('gravedad', lambda x: x.isin(['grave','fallecido']).sum()),
         n_peatones=('tipo_vru', lambda x: (x=='Peatón').sum()),
         n_ciclistas=('tipo_vru', lambda x: (x=='Ciclista').sum()))
    .reset_index()
    .sort_values(['distrito','n_accidentes'], ascending=[True, False])
)
# Top 10 por distrito
calles_top10 = calles.groupby('distrito').head(10)
calles_top10.to_csv(f'{pbi_path}/pbi_calles_top3.csv', index=False, encoding='utf-8-sig')

# Fallecidos por edad
fallecidos = df_top3[df_top3['gravedad'] == 'fallecido'].copy()
edad_fall = (
    fallecidos.groupby(['distrito', 'rango_edad', 'tipo_vru'])
    ['num_expediente'].count()
    .reset_index()
    .rename(columns={'num_expediente': 'n_fallecidos'})
)
edad_fall.to_csv(f'{pbi_path}/pbi_fallecidos_edad_top3.csv', index=False, encoding='utf-8-sig')

print('✅ pbi_horario_top3.csv guardado')
print('✅ pbi_diasemana_top3.csv guardado')
print('✅ pbi_calles_top3.csv guardado')
print('✅ pbi_fallecidos_edad_top3.csv guardado')

✅ pbi_horario_top3.csv guardado
✅ pbi_diasemana_top3.csv guardado
✅ pbi_calles_top3.csv guardado
✅ pbi_fallecidos_edad_top3.csv guardado


In [9]:
# ══════════════════════════════════════════════════════════
# CELDA 9 — Resumen final de ficheros generados
# ══════════════════════════════════════════════════════════

import os
archivos = os.listdir(pbi_path)
print('=== FICHEROS LISTOS PARA POWER BI ===\n')
for f in sorted(archivos):
    size = os.path.getsize(f'{pbi_path}/{f}')
    print(f'  {f:<45} {size/1024:.1f} KB')

print(f'\n✅ Total: {len(archivos)} ficheros en {pbi_path}')
print('\n=== INSTRUCCIONES DE IMPORTACIÓN ===')
print('1. Descarga la carpeta PowerBI completa desde Google Drive')
print('2. En Power BI Desktop → Obtener datos → JSON (para GeoJSON)')
print('3. En Power BI Desktop → Obtener datos → CSV (para el resto)')
print('4. El GeoJSON se usa en el visual de mapa ArcGIS o Shape Map')

=== FICHEROS LISTOS PARA POWER BI ===

  distritos_madrid.geojson                      396.3 KB
  distritos_madrid_topo.json                    264.7 KB
  pbi_calles_top3.csv                           1.3 KB
  pbi_diasemana_top3.csv                        1.2 KB
  pbi_fallecidos_edad_top3.csv                  0.5 KB
  pbi_graves_edad_top3.csv                      2.1 KB
  pbi_horario_top3.csv                          3.3 KB
  pbi_ipd_temporal.csv                          5.6 KB
  pbi_score_ranking.csv                         2.5 KB
  pbi_validacion_irp.csv                        1.3 KB

✅ Total: 10 ficheros en /content/drive/MyDrive/TFM_Seguridad_Vial/outputs/PowerBI

=== INSTRUCCIONES DE IMPORTACIÓN ===
1. Descarga la carpeta PowerBI completa desde Google Drive
2. En Power BI Desktop → Obtener datos → JSON (para GeoJSON)
3. En Power BI Desktop → Obtener datos → CSV (para el resto)
4. El GeoJSON se usa en el visual de mapa ArcGIS o Shape Map


In [10]:
# ══════════════════════════════════════════════════════════
# Convertir GeoJSON a TopoJSON para Power BI Shape Map
# ══════════════════════════════════════════════════════════

!pip install topojson --quiet

import topojson
import geopandas as gpd
import json

outputs  = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs'
pbi_path = f'{outputs}/PowerBI'

# Cargamos el GeoJSON
gdf = gpd.read_file(f'{pbi_path}/distritos_madrid.geojson')

# Convertimos a TopoJSON
topo = topojson.Topology(gdf, prequantize=False)
topo.to_json(f'{pbi_path}/distritos_madrid_topo.json')

print('✅ TopoJSON guardado')
print(f'Distritos: {len(gdf)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 kB 1.5 MB/s eta 0:00:00
✅ TopoJSON guardado
Distritos: 21


In [11]:
# ══════════════════════════════════════════════════════════
# CELDA EXTRA — Graves + Fallecidos por edad Top 3
# ══════════════════════════════════════════════════════════

TOP3 = ['CENTRO', 'CHAMBERÍ', 'SALAMANCA']

graves_fall = df_top3[df_top3['gravedad'].isin(['grave', 'fallecido'])].copy()

edad_graves = (
    graves_fall.groupby(['distrito', 'rango_edad', 'gravedad'])
    ['num_expediente'].count()
    .reset_index()
    .rename(columns={'num_expediente': 'n_victimas'})
)

edad_graves.to_csv(f'{pbi_path}/pbi_graves_edad_top3.csv',
                   index=False, encoding='utf-8-sig')
print(f'✅ pbi_graves_edad_top3.csv guardado — {edad_graves.shape}')
print(edad_graves.groupby('distrito')['n_victimas'].sum())

✅ pbi_graves_edad_top3.csv guardado — (62, 4)
distrito
CENTRO       94
CHAMBERÍ     83
SALAMANCA    95
Name: n_victimas, dtype: int64
